# CRUD Operations in ChromaDB (Adding and Querying Documents)**Level:** Scratch → Intermediate**Runs on:** Google Colab (Free CPU)**CRUD** = **C**reate, **R**ead, **U**pdate, **D**elete — the four basic operations every database supports.### What you will learn1. **Create** — set up a client and collection2. **Create (Add)** — insert documents with ids and metadata3. **Read (Query)** — semantic search + get-by-id + filters4. **Update** — modify existing documents/metadata5. **Delete** — remove documents by id or by filter6. Checking collection state (`count`, `peek`, `list_collections`)

### 1. Install ChromaDBRun this once.

In [ ]:
!pip install -q chromadb

### 2. CREATE — Client and Collection

In [ ]:
import chromadb# In-memory client (resets when runtime restarts) - great for learningclient = chromadb.Client()# Create a fresh collection to practice CRUD operations oncollection = client.create_collection(name="crud_demo")print("Collection created:", collection.name)print("Current document count:", collection.count())

### 3. CREATE — Adding Documents (`add`)Each document needs a unique `id`. `metadata` and `documents` are optional extras that describe each entry.

In [ ]:
collection.add(    documents=[        "Python is a popular programming language for AI.",        "The weather today is sunny and warm.",        "Machine learning models learn patterns from data.",        "I enjoy playing football on weekends.",        "Deep learning uses neural networks with many layers."    ],    metadatas=[        {"category": "tech"},        {"category": "weather"},        {"category": "tech"},        {"category": "sports"},        {"category": "tech"}    ],    ids=["doc1", "doc2", "doc3", "doc4", "doc5"])print("Documents added! Total count:", collection.count())

**Discussion Point:** If you try to `add()` an `id` that already exists, ChromaDB will raise an error. Use `upsert()` (shown later) if you want "add or update" behavior instead.

### 4. READ — Peek at the Collection`peek()` shows a quick sample of what's inside, without needing a query.

In [ ]:
preview = collection.peek(limit=3)for doc_id, doc in zip(preview["ids"], preview["documents"]):    print(doc_id, "->", doc)

### 5. READ — Get Documents by IDUse `get()` when you know exactly which id(s) you want — this is an exact lookup, not a similarity search.

In [ ]:
result = collection.get(ids=["doc1", "doc3"])for doc_id, doc, meta in zip(result["ids"], result["documents"], result["metadatas"]):    print(f"{doc_id} | {meta} | {doc}")

### 6. READ — Semantic Search (`query`)This is the core "vector database" feature: search by **meaning**, not exact words.

In [ ]:
results = collection.query(    query_texts=["Tell me about AI and neural networks"],    n_results=2)for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):    print(f"{doc_id} | distance={distance:.4f} | {doc}")

### 7. READ — Filtering with `where` (Metadata Filter)Combine semantic search with exact filters — e.g., "only search within the 'tech' category".

In [ ]:
results = collection.query(    query_texts=["something fun to do outside"],    n_results=3,    where={"category": "sports"})for doc_id, doc, distance in zip(results["ids"][0], results["documents"][0], results["distances"][0]):    print(f"{doc_id} | distance={distance:.4f} | {doc}")

### 8. UPDATE — Modifying Existing Documents`update()` changes the text and/or metadata of documents that **already exist** (matched by `id`).

In [ ]:
collection.update(    ids=["doc4"],    documents=["I enjoy playing football and basketball on weekends."],    metadatas=[{"category": "sports", "updated": True}])updated = collection.get(ids=["doc4"])print(updated["documents"])print(updated["metadatas"])

### 9. UPSERT — Add or Update in One Step`upsert()` is useful when you're not sure if the `id` already exists: if it exists, it updates it; if not, it creates it.

In [ ]:
collection.upsert(    documents=[        "Deep learning uses neural networks with many layers and GPUs.",  # existing id -> will update        "The Eiffel Tower is located in Paris."                           # new id -> will create    ],    ids=["doc5", "doc6"])print("Total count after upsert:", collection.count())print(collection.get(ids=["doc5", "doc6"])["documents"])

### 10. DELETE — Removing Documents by ID

In [ ]:
collection.delete(ids=["doc2"])print("Total count after delete:", collection.count())print("Remaining ids:", collection.get()["ids"])

### 11. DELETE — Removing Documents by FilterYou can also delete every document that matches a metadata filter, without listing ids manually.

In [ ]:
collection.delete(where={"category": "tech"})print("Total count after category delete:", collection.count())print("Remaining ids:", collection.get()["ids"])

### 12. Verifying Everything (Final State Check)

In [ ]:
final_state = collection.get()print("Final document count:", collection.count())for doc_id, doc, meta in zip(final_state["ids"], final_state["documents"], final_state["metadatas"]):    print(f"{doc_id} | {meta} | {doc}")

### 13. Key Takeaways (Recap for Students)| Operation | Method | Notes ||---|---|---|| Create | `client.create_collection()` | Makes a new collection (errors if it already exists) || Create/Add | `collection.add()` | Insert new documents; errors on duplicate `id` || Read (exact) | `collection.get()` | Fetch by `id`, no similarity involved || Read (semantic) | `collection.query()` | Search by meaning, returns closest matches + distances || Read (preview) | `collection.peek()` | Quick look at a few stored documents || Update | `collection.update()` | Modify existing document/metadata by `id` || Add or Update | `collection.upsert()` | Creates if missing, updates if it exists || Delete | `collection.delete()` | Remove by `id` list or by `where` metadata filter |### 14. Practice Exercises1. Add 5 new documents, then use `upsert()` to update 2 of them and create 1 brand-new one in a single call.2. Delete all documents where `category` equals a value of your choice, then confirm with `collection.count()`.3. Write a small function `safe_add(collection, id, text)` that uses `upsert()` internally so it never raises a duplicate-id error.